In [10]:
!pip install flask pyngrok
from google.colab import drive
drive.mount('/content/drive')
import pickle
import pandas as pd
import numpy as np
from flask import Flask, request, jsonify
from pyngrok import ngrok  # Use pyngrok instead of flask-ngrok

# Load the pipeline and model
def load_pipeline(pipeline_path):
    with open(pipeline_path, 'rb') as file:
        return pickle.load(file)

def load_model(model_path):
    with open(model_path, 'rb') as file:
        return pickle.load(file)

# Load pipeline and model
pipeline_path = '/content/drive/MyDrive/Colab Notebooks/Risk Prediction/full_pipeline.pkl'
model_path = '/content/drive/MyDrive/Colab Notebooks/Risk Prediction/best_xgb_model.pkl'

loaded_pipeline = load_pipeline(pipeline_path)
loaded_xgb_model = load_model(model_path)

# Function to preprocess input data
def preprocess_data(data):
    df = pd.DataFrame([data])  # Convert input to DataFrame
    transformed_df = loaded_pipeline.transform(df)  # Apply pipeline transformation
    return transformed_df

# Create Flask app
app = Flask(__name__)

@app.route('/')
def home():
    return "Welcome to the Risk Prediction API!"

@app.route('/predict', methods=['POST'])
def predict():
    try:
        # Get JSON data
        data = request.get_json(force=True)
        # Preprocess input data
        preprocessed_data = preprocess_data(data)
        # Make prediction
        prediction = loaded_xgb_model.predict(preprocessed_data)
        # Adjust prediction if necessary
        prediction_adjusted = prediction + 1
        # Return result as JSON
        return jsonify({'prediction': int(prediction_adjusted[0])})
    except Exception as e:
        return jsonify({'error': str(e)})

# Start ngrok tunnel
public_url = ngrok.connect(5000).public_url
print(f"Public URL: {public_url}")

# Run the Flask app
app.run(port=5000)




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


ERROR:pyngrok.process.ngrok:t=2025-03-25T08:03:39+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2025-03-25T08:03:39+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2025-03-25T08:03:39+0000 lvl=eror msg="terminating with error" obj=app err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your aut

PyngrokNgrokError: The ngrok process errored on start: authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.

In [9]:
if __name__ == '__main__':
    app.run()


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
Exception in thread Thread-11:
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/usr/local/lib/python3.11/dist-packages/urllib3/util/connection.py", line 73, in create_connection
    sock.connect(sa)
ConnectionRefusedError: [Errno 111] Connection refused

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/urllib3/connectionpool.py", line 787, in urlopen
    response = self._make_request(
           

In [ ]:

import numpy as np
import pickle
import pandas as pd

# Load the original data
raw_data_path = '/content/drive/MyDrive/Colab Notebooks/Risk Prediction/train.csv'
raw_df = pd.read_csv(raw_data_path, index_col='Id')

# Function to generate fake data
def generate_fake_data(df, num_samples):
    fake_data = {}

    for column in df.columns:
        if df[column].dtype == 'object':
            # For categorical columns, sample from the unique values
            fake_data[column] = np.random.choice(df[column].unique(), num_samples)
        elif df[column].dtype == 'int64':
            # For integer columns, sample from the range of values and convert to int
            fake_data[column] = np.random.randint(df[column].min(), df[column].max() + 1, num_samples)
        elif df[column].dtype == 'float64':
            # For float columns, sample from the range of values
            fake_data[column] = np.random.uniform(df[column].min(), df[column].max(), num_samples)

    # Convert the dictionary to a DataFrame
    fake_data_df = pd.DataFrame(fake_data)

    return fake_data_df

# Generate fake data
num_samples = 100  # Number of fake samples to generate
fake_data_df = generate_fake_data(raw_df, num_samples)

# Ensure the fake data has the same columns as the original data
fake_data_df = fake_data_df[raw_df.columns]


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.11/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['Medical_History_10']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


In [ ]:

# Example of how to load and use the full pipeline
def load_pipeline(pipeline_path):
    with open(pipeline_path, 'rb') as file:
        loaded_pipeline = pickle.load(file)
    return loaded_pipeline

# Load the full pipeline
loaded_pipeline = load_pipeline('/content/drive/MyDrive/Colab Notebooks/Risk Prediction/full_pipeline.pkl')
transformed_df = loaded_pipeline.transform(fake_data_df)


# Load the saved XGBoost model
model_path = '/content/drive/MyDrive/Colab Notebooks/Risk Prediction/best_xgb_model.pkl'
with open(model_path, 'rb') as file:
    loaded_xgb_model = pickle.load(file)


predictions_adjusted = loaded_xgb_model.predict(transformed_df)
predictions_adjusted += 1

In [ ]:
predictions_adjusted

array([5, 5, 8, 5, 8, 5, 6, 6, 6, 6, 6, 8, 8, 4, 6, 6, 6, 5, 8, 5, 8, 6,
       6, 8, 8, 6, 5, 3, 6, 7, 8, 8, 5, 8, 8, 6, 5, 6, 5, 8, 5, 6, 8, 6,
       1, 8, 5, 5, 6, 6, 8, 8, 7, 8, 6, 6, 8, 8, 7, 6, 8, 5, 7, 4, 8, 6,
       8, 5, 5, 6, 6, 6, 6, 5, 8, 8, 6, 8, 8, 5, 8, 6, 8, 5, 5, 8, 6, 8,
       6, 8, 4, 5, 6, 5, 8, 6, 5, 5, 7, 6], dtype=int32)

In [ ]:
from flask import Flask, request, jsonify
import pickle
import pandas as pd
from xgboost import XGBClassifier

app = Flask(__name__)



@app.route('/batch_predict', methods=['POST'])
def batch_predict():
    # Get the JSON data from the request
    data = request.get_json()

    # Convert the JSON data to a DataFrame
    input_data = pd.DataFrame(data)

    # Preprocess the input data
    input_data_transformed = loaded_pipeline.transform(input_data)

    # Make predictions
    predictions_adjusted = loaded_xgb_model.predict(input_data_transformed)

    # Adjust the predictions back to the original class range
    predictions = predictions_adjusted + 1

    # Return the predictions as JSON
    return jsonify(predictions.tolist())

if __name__ == '__main__':
    app.run(debug=True)

 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug: * Restarting with stat


Docker File

In [ ]:
# Use an official Python runtime as a parent image
FROM python:3.9-slim  # Ensure this matches the Python version in Colab

# Set the working directory in the container
WORKDIR /app

# Copy the current directory contents into the container at /app
COPY . /app

# Copy the requirements file from your local machine to the container
COPY requirements_colab.txt /app/requirements.txt

# Install any needed packages specified in requirements.txt
RUN pip install --no-cache-dir -r requirements.txt

# Make port 5000 available to the world outside this container
EXPOSE 5000

# Run app.py when the container launches
CMD ["python", "app.py"]